In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
sample=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sample.to_csv("submission.csv",index=False)

In [3]:
train_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
train_df.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


# MILESTONE 1

### Q1)

In [7]:
train_df["answer"].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

### Q2)

In [8]:
cleaned=train_df["prompt"].str.lower()

In [9]:
import string

In [10]:
for char in string.punctuation:
    cleaned=cleaned.str.replace(char,"",regex=False)

In [11]:
len(cleaned.str.split().explode().unique())

859

### Q3)

In [12]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [13]:
words = cleaned.iloc[1].split()
filtered = [word for word in words if word not in ENGLISH_STOP_WORDS]
len(filtered)

3

### Q4)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:
combined = train_df[["prompt", "A", "B", "C", "D", "E"]].apply(lambda row: " ".join(row), axis=1)

In [16]:
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit_transform(combined)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 64180 stored elements and shape (2000, 2762)>

In [17]:
len(vectorizer.vocabulary_)

2762

### Q5)

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
prompt_vec = vectorizer.transform([train_df.loc[1, "prompt"]])
option_a_vec = vectorizer.transform([train_df.loc[1, "A"]])

In [20]:
score = cosine_similarity(prompt_vec, option_a_vec)
round(float(score[0][0]), 4)

0.5081

### Q6)

In [21]:
correct = 0
for idx, row in train_df.iterrows():
    prompt_vec = vectorizer.transform([row["prompt"]])
    scores = {}
    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        scores[option] = cosine_similarity(prompt_vec, option_vec)[0][0]
    best_option = max(scores, key=lambda x: scores[x])
    if best_option == row["answer"]:
        correct += 1

In [22]:
percentage = round((correct / len(train_df)) * 100, 2)
print(percentage)

13.55


### Q7)

In [23]:
from collections import Counter

In [24]:
top3 = train_df["answer"].value_counts().index[:3].tolist()

In [25]:
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1 / (predicted.index(actual) + 1)
    return 0

In [26]:
scores = train_df["answer"].apply(lambda x: apk(x, top3))
map3 = round(scores.mean(), 4)
print(map3)

0.4212


### Q8)

In [27]:
map3_scores = []

In [28]:
for idx, row in train_df.iterrows():
    prompt_vec = vectorizer.transform([row["prompt"]])
    sim_scores = {}
    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sim_scores[option] = cosine_similarity(prompt_vec, option_vec)[0][0]
    top3_preds = sorted(sim_scores, key=lambda x: sim_scores[x], reverse=True)[:3]
    map3_scores.append(apk(row["answer"], top3_preds))

In [29]:
print(round(sum(map3_scores) / len(map3_scores), 4))

0.2962
